# LangChain OpenAI Chatbot

This notebook demonstrates how to build a chatbot using LangChain and OpenAI.

## Setup

First, let's import the necessary libraries and load environment variables.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

# Load environment variables from .env file
load_dotenv()

print("✅ Libraries imported successfully!")

## 1. Basic Chatbot (No Memory)

Let's start with a simple chatbot that responds to individual messages without remembering previous context.

In [ ]:
# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Create a simple prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Be concise and friendly."),
    ("user", "{input}")
])

# Create a chain
chain = prompt | llm | StrOutputParser()

print("✅ Basic chatbot initialized!")

In [ ]:
# Test the basic chatbot
response = chain.invoke({"input": "Hello! What can you help me with?"})
print(response)

In [ ]:
# Try another message (notice it won't remember the previous conversation)
response = chain.invoke({"input": "What did I just ask you?"})
print(response)

## 2. Chatbot with Conversation Memory

Now let's build a chatbot that remembers the conversation history.

In [ ]:
# Initialize the LLM
llm_with_memory = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Create a prompt template with message history
prompt_with_memory = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Be concise and friendly. Remember the conversation context."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}")
])

# Create the chain
chat_chain = prompt_with_memory | llm_with_memory | StrOutputParser()

# Initialize conversation history
chat_history = []

print("✅ Chatbot with memory initialized!")

In [ ]:
def chat(user_input):
    """Send a message to the chatbot and get a response."""
    global chat_history
    
    # Get response from the chain
    response = chat_chain.invoke({
        "chat_history": chat_history,
        "input": user_input
    })
    
    # Update chat history
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))
    
    return response

def reset_chat():
    """Reset the conversation history."""
    global chat_history
    chat_history = []
    print("🔄 Conversation history reset!")

print("✅ Helper functions defined!")

### Test the Chatbot with Memory

In [ ]:
# Start a conversation
response = chat("Hi! My name is Alice.")
print(f"AI: {response}")

In [ ]:
# Continue the conversation
response = chat("What's my name?")
print(f"AI: {response}")

In [ ]:
# Ask another question
response = chat("Can you tell me about LangChain?")
print(f"AI: {response}")

In [ ]:
# Follow-up question (it should remember the context)
response = chat("What are its main use cases?")
print(f"AI: {response}")

### View Conversation History

In [ ]:
# View the full conversation history
print("📝 Conversation History:\n")
for i, message in enumerate(chat_history, 1):
    role = "User" if isinstance(message, HumanMessage) else "AI"
    print(f"{i}. {role}: {message.content}\n")

## 3. Interactive Chatbot Loop

Let's create an interactive chat session where you can have a continuous conversation.

In [ ]:
def interactive_chat():
    """Start an interactive chat session."""
    print("🤖 Chatbot ready! Type 'quit' to exit, 'reset' to clear history.\n")
    
    while True:
        user_input = input("You: ").strip()
        
        if user_input.lower() == 'quit':
            print("👋 Goodbye!")
            break
        
        if user_input.lower() == 'reset':
            reset_chat()
            continue
        
        if not user_input:
            continue
        
        response = chat(user_input)
        print(f"\nAI: {response}\n")

# Uncomment to start interactive chat
# interactive_chat()

## 4. Specialized Chatbot Examples

Let's create chatbots with different personalities and purposes.

In [ ]:
# Code Assistant Chatbot
code_assistant_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert programming assistant. Help users with coding questions, debugging, and best practices. Provide clear code examples."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}")
])

code_assistant_chain = code_assistant_prompt | llm_with_memory | StrOutputParser()
code_chat_history = []

def code_assistant(user_input):
    global code_chat_history
    response = code_assistant_chain.invoke({
        "chat_history": code_chat_history,
        "input": user_input
    })
    code_chat_history.append(HumanMessage(content=user_input))
    code_chat_history.append(AIMessage(content=response))
    return response

print("✅ Code Assistant chatbot created!")

In [ ]:
# Test the Code Assistant
response = code_assistant("How do I create a list comprehension in Python?")
print(response)

## 5. Streaming Responses

For a better user experience, we can stream responses token by token.

In [ ]:
def chat_with_streaming(user_input):
    """Send a message and stream the response."""
    global chat_history
    
    print("AI: ", end="", flush=True)
    
    full_response = ""
    for chunk in chat_chain.stream({
        "chat_history": chat_history,
        "input": user_input
    }):
        print(chunk, end="", flush=True)
        full_response += chunk
    
    print()  # New line after streaming
    
    # Update chat history
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=full_response))
    
    return full_response

print("✅ Streaming function defined!")

In [ ]:
# Test streaming (reset chat first for a clean test)
reset_chat()
chat_with_streaming("Tell me an interesting fact about artificial intelligence.")

## Summary

In this notebook, we've covered:

1. **Basic Chatbot** - Simple question-answer without memory
2. **Chatbot with Memory** - Maintains conversation context
3. **Interactive Chat Loop** - Continuous conversation interface
4. **Specialized Chatbots** - Different personalities/purposes (e.g., Code Assistant)
5. **Streaming Responses** - Real-time token-by-token output

### Next Steps

- Add conversation summarization for long chats
- Implement conversation persistence (save/load)
- Add retrieval-augmented generation (RAG) for knowledge-based responses
- Integrate with external tools and APIs
- Add conversation analytics and sentiment analysis